# PGx Risk Calculator Dashboard – Full Deployment Workflow

**Purpose:** Deploy the PGx Risk Calculator Dashboard from cohorts with aggregated feature importances through Lambda/Docker.  
**Updated:** January–February 2026

**Overview**

This notebook **consolidates the pipeline from Step 4 onward**. It **runs pipeline Step 4, 5, and 6 in this notebook** (see cells below), then prepares and deploys the dashboard:

- **Pipeline Step 4** — Model data: `4_model_data/create_model_data.py` (model_events.parquet per cohort/age_band)
- **Pipeline Step 5** — PGx analysis: `5_pgx_analysis/run_analysis.py` (PGx features added to model data)
- **Pipeline Step 6** — Final model training: `6_final_model/run_final_model.py` (trained models and feature_schema.json)

**Required inputs (must exist before running pipeline Step 4):**

- **Cohorts** (Step 2) — `gold/cohorts` cohort.parquet files (case/control and target dates)  
- **Feature importances** (Step 3/3b) — cohort_feature_importance CSVs and feature_filtering_summary.json, synced from S3 or under project/NVMe

**Required outputs (for deployment):**

- **SHAP** (Step 7) and **FFA** (Step 8) results — must be produced and combined for the Causal Analysis tab.

Cohort / model mapping

| Model | PGx cohort | Age bands | Description |
|-------|------------|-----------|-------------|
| **Opioid ED** | `opioid_ed` | 13-24, 25-44, 45-54, 55-64 | Opioid-related ED visit predictive model |
| **Polypharmacy** | `non_opioid_ed` | 65-74, 75-84, 85-94 | Polypharmacy / adverse drug event model |

## Workflow Steps

1. **Sync inputs from S3 to NVMe** (idempotent) – Step 3a/3b feature importance (and optionally Step 6 if already built elsewhere).
2. **Verify inputs** – Feature importance (Step 3/3b) per cohort/age_band; Step 6 outputs if already present.
3. **Pipeline Phase 4–6** – Run the **Pipeline Phase 4**, **5**, and **6** cells below (model data → PGx analysis → final model training). Skip if Phase 6 outputs already exist and are synced.
4. **Step 1: Generate metadata** (idempotent, checkpoint) – Extract valid codes from feature importance for dashboard dropdowns.
5. **Step 2: Combine SHAP/FFA** (required) – Run Step 7 (SHAP) and Step 8 (FFA), then combine results for Causal Analysis tab.
6. **Results inspection** (optional) – View top causal/consensus features and combined importance per cohort/age_band.
7. **Feature importance display** (optional) – View top-N features from Step 6 or Step 3b per cohort/age_band.
8. **Step 3: Prepare models** (idempotent, checkpoint) – Package models and feature schemas from Step 6 outputs.
9. **Step 4: Prepare Lambda directory** – Assemble `lambda_dir` for Docker build.
10. **Step 5: Verify Lambda directory** – Ensure required files are present.
11. **Verify infrastructure** – Docker, ECR, and API Gateway checks for PGx dashboard.
12. **Step 6: Build and deploy** – Build Docker image, push to ECR, update Lambda/API Gateway.

## Reference

- PGx data prep: `9_risk_dashboard/data_preparation/`

In [5]:
# Setup: paths and project root
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "9_risk_dashboard":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "9_risk_dashboard").exists():
    pass
else:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import sync_s3_to_local, check_step_checkpoint_exists, save_step_checkpoint

DASHBOARD_DIR = PROJECT_ROOT / "9_risk_dashboard"
DATA_PREP_DIR = DASHBOARD_DIR / "data_preparation"
DEPLOY_DIR = DASHBOARD_DIR / "deployment"
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

print("PGx Risk Calculator Workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Dashboard dir: {DASHBOARD_DIR}")
print(f"Data prep: {DATA_PREP_DIR}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print("=" * 60)

PGx Risk Calculator Workflow
Project root: /home/pgx3874/pgx-analysis
Dashboard dir: /home/pgx3874/pgx-analysis/9_risk_dashboard
Data prep: /home/pgx3874/pgx-analysis/9_risk_dashboard/data_preparation
Data root (NVMe/local): /mnt/nvme


In [6]:
# Configuration: PGx cohorts and age bands (aligned with prepare_lambda_dir.py)
# opioid_ed (younger age bands); non_opioid_ed / polypharmacy (older age bands)
REQUIRED_COHORTS = {
    "opioid_ed": ["13-24", "25-44", "45-54", "55-64"],
    "non_opioid_ed": ["65-74", "75-84", "85-94"],
}

# Input dirs (required for pipeline Step 4–6)
# Cohorts: Step 2 cohort.parquet files (create_model_data reads case/control and target dates from here).
COHORTS_ROOT = DATA_ROOT / "gold" / "cohorts"
# Feature importance: Step 3/3b outputs — cohort_feature_importance.csv and feature_filtering_summary.json per cohort/age_band.
FI_ROOT = DATA_ROOT / "gold" / "feature_importance"
STEP3_OUTPUTS = STEP3B_OUTPUTS = FI_ROOT
# Model data: single canonical location (Step 4 output, Step 5/6 input).
from py_helpers.env_utils import get_model_data_root
MODEL_DATA_ROOT = get_model_data_root()

# Output dirs (Step 6 final model outputs; data prep and Lambda read from these)
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_OUTPUTS_ALT = DATA_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_GOLD = DATA_ROOT / "gold" / "final_model"  # S3 layout: cohort/13-24/*.joblib

print("Cohorts and age bands:")
for cohort, bands in REQUIRED_COHORTS.items():
    print(f"  {cohort}: {bands}")
print("\nInput dirs (for Step 4–6):")
print(f"  Cohorts (2):              {COHORTS_ROOT}")
print(f"  Feature importance (3/3b): {FI_ROOT}  (CSVs + feature_filtering_summary.json)")
print(f"  Model data (4; in/out):   {MODEL_DATA_ROOT}")
print("\nOutput dirs (Step 6):")
print(f"  Project:   {FINAL_MODEL_OUTPUTS}")
print(f"  NVMe:      {FINAL_MODEL_OUTPUTS_ALT}")
print(f"  gold/NVMe: {FINAL_MODEL_GOLD}")

Cohorts and age bands:
  opioid_ed: ['13-24', '25-44', '45-54', '55-64']
  non_opioid_ed: ['65-74', '75-84', '85-94']

Input dirs (for Step 4–6):
  Cohorts (2):              /mnt/nvme/gold/cohorts
  Feature importance (3/3b): /mnt/nvme/gold/feature_importance  (CSVs + feature_filtering_summary.json)
  Model data (4; in/out):   /mnt/nvme/4_model_data

Output dirs (Step 6):
  Project:   /home/pgx3874/pgx-analysis/6_final_model/outputs
  NVMe:      /mnt/nvme/6_final_model/outputs
  gold/NVMe: /mnt/nvme/gold/final_model


## Sync required inputs from S3 to NVMe (idempotent)

Sync **cohorts** (Step 2), **feature importance** (Step 3/3b), and **Step 6** final model outputs from S3 so pipeline and data preparation can read from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [7]:
# Sync cohorts (Step 2), Step 3a/3b feature importance, and Step 6 final models from S3 to NVMe (DATA_ROOT).
# Cohorts -> COHORTS_ROOT (gold/cohorts); Feature importance -> gold/feature_importance; Step 6 -> gold/final_model.
COHORTS_ROOT.mkdir(parents=True, exist_ok=True)
FI_SYNC_TARGET = DATA_ROOT / "gold" / "feature_importance"
FI_SYNC_TARGET.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_GOLD.mkdir(parents=True, exist_ok=True)

sync_s3_to_local(f"s3://{S3_BUCKET}/gold/cohorts/", COHORTS_ROOT, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/feature_importance/", FI_SYNC_TARGET, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/final_model/", FINAL_MODEL_GOLD, profile=AWS_PROFILE)
print("Sync complete. Run Step 0 verification below.")

Sync complete. Run Step 0 verification below.


## Step 0: Verify inputs (FI required; 4_model_data and Step 6 informational)

**Required:** **Feature importance** (Step 3/3b) — must exist for each cohort/age_band so Pipeline Step 4 can run.

**Informational:** **ModelData** checks `DATA_ROOT/4_model_data` and `PROJECT_ROOT/4_model_data` (same location `create_model_data.py` writes to). **Model** = Step 6 outputs. Both are produced by Pipeline Step 4–6 cells below; if already present, you can skip those cells.

In [8]:
def check_feature_importance(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # Step 3b refined: FI_ROOT (NVMe) then project 3b/outputs
    for base in (STEP3B_OUTPUTS, PROJECT_ROOT / "3b_feature_importance_eda" / "outputs"):
        fi_3b = base / cohort / ab / f"{cohort}_{ab}_cohort_feature_importance.csv"
        if fi_3b.exists():
            return True
    # Step 3 aggregated: FI_ROOT then project 3a/outputs
    for base in (STEP3_OUTPUTS, PROJECT_ROOT / "3a_feature_importance" / "outputs"):
        fi_3 = base / cohort / ab / f"{cohort}_{ab}_aggregated_feature_importance.csv"
        if fi_3.exists():
            return True
    return False

def check_cohorts(cohort: str, age_band: str) -> bool:
    """Check Step 2 cohort.parquet exists for at least one year (2016–2019). Layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet."""
    for year in (2016, 2017, 2018, 2019):
        p = COHORTS_ROOT / f"cohort_name={cohort}" / f"event_year={year}" / f"age_band={age_band}" / "cohort.parquet"
        if p.exists():
            return True
    return False

def check_model_data(cohort: str, age_band: str) -> bool:
    """Check model_events.parquet at canonical MODEL_DATA_ROOT (same location create_model_data.py writes to)."""
    p = MODEL_DATA_ROOT / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
    return p.exists()

def check_final_model(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # 1) Project or DATA_ROOT/6_final_model/outputs: cohort/13_24/models/*.joblib
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        model_dir = base / cohort / ab
        if not model_dir.exists():
            continue
        models_sub = model_dir / "models"
        if models_sub.exists() and any(models_sub.glob("*.joblib")):
            return True
        if (model_dir / "feature_schema.json").exists():
            return True
    # 2) DATA_ROOT/gold/final_model (S3-synced): cohort/13-24/*.joblib (hyphen in age_band)
    gold_dir = FINAL_MODEL_GOLD / cohort / age_band
    if gold_dir.exists() and any(gold_dir.glob("*.joblib")):
        return True
    return False

print("Step 0: Verify feature importance (required); cohorts and 4_model_data (Step 4 inputs); Step 6 (informational)\n")
print("  Locations: Cohorts=COHORTS_ROOT, FI=Step 3/3b, ModelData=MODEL_DATA_ROOT, Model=Step 6 outputs\n")
fi_ok_all = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        cohorts_ok = check_cohorts(cohort, age_band)
        fi_ok = check_feature_importance(cohort, age_band)
        model_data_ok = check_model_data(cohort, age_band)
        model_ok = check_final_model(cohort, age_band)
        if not fi_ok:
            fi_ok_all = False
        status = "ready" if fi_ok else "missing FI"
        print(f"  {cohort} / {age_band}:  Cohorts={cohorts_ok}, FI={fi_ok}, ModelData={model_data_ok}, Model={model_ok}  -> {status}")
if fi_ok_all:
    print("\nAll prerequisites are available to build model data. Run Pipeline Step 4–6 cells below.")
    print("  (If Step 6 is already built elsewhere, you can sync from S3 or skip those cells.)")
else:
    print("\nMissing feature importance for some cohort/age_band. Sync from S3 or run Step 3/3b first, then re-run this cell.")
if fi_ok_all:
    cohorts_missing = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands if not check_cohorts(c, ab)]
    if cohorts_missing:
        print("\nCohorts=False for some cohort/age_band. Sync gold/cohorts from S3 (run Sync cell) or run Step 2. Expected layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016–2019).")

Step 0: Verify feature importance (required); cohorts and 4_model_data (Step 4 inputs); Step 6 (informational)

  Locations: Cohorts=COHORTS_ROOT, FI=Step 3/3b, ModelData=MODEL_DATA_ROOT, Model=Step 6 outputs

  opioid_ed / 13-24:  Cohorts=True, FI=True, ModelData=True, Model=True  -> ready
  opioid_ed / 25-44:  Cohorts=True, FI=True, ModelData=True, Model=True  -> ready
  opioid_ed / 45-54:  Cohorts=True, FI=True, ModelData=True, Model=True  -> ready
  opioid_ed / 55-64:  Cohorts=True, FI=True, ModelData=True, Model=True  -> ready
  non_opioid_ed / 65-74:  Cohorts=True, FI=True, ModelData=True, Model=True  -> ready
  non_opioid_ed / 75-84:  Cohorts=True, FI=True, ModelData=True, Model=True  -> ready
  non_opioid_ed / 85-94:  Cohorts=True, FI=True, ModelData=True, Model=True  -> ready

All prerequisites are available to build model data. Run Pipeline Step 4–6 cells below.
  (If Step 6 is already built elsewhere, you can sync from S3 or skip those cells.)


# Pipeline Phase 4: Model data

Build `model_events.parquet` for each cohort/age_band from Step 2 cohort data and Step 3b feature importance. Outputs go to `MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet`. Run the cell below for all PGx cohorts/age_bands defined in this notebook.

In [ ]:
# Pipeline Step 4: BUILD model_events.parquet by running create_model_data.py, then QA.
# The script READS: COHORTS_ROOT (cohort.parquet), gold/medical, gold/pharmacy, and feature importance.
# It WRITES: MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet
import duckdb

def _model_data_candidates(cohort: str, age_band: str):
    """Canonical location for model_events.parquet (Step 4 writes to MODEL_DATA_ROOT)."""
    return [MODEL_DATA_ROOT]

def _model_data_path(cohort: str, age_band: str) -> Path:
    """Resolve model_events.parquet path (Step 4 writes to get_model_data_root() = DATA_ROOT or PROJECT on Linux)."""
    for base in _model_data_candidates(cohort, age_band):
        p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
        if p.exists():
            return p
    return None

def _log_model_data_qa(cohort: str, age_band: str) -> None:
    """Log location, target distribution, and control:case ratio for model_events.parquet."""
    path = _model_data_path(cohort, age_band)
    if not path:
        print(f"  [WARN] model_events.parquet not found for {cohort}/{age_band}")
        for base in _model_data_candidates(cohort, age_band):
            p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
            print(f"    Checked: {p}  (exists: {p.exists()})")
        print(f"    Build did not write output. Check script stdout above: [INFO] data roots and example cohort path (exists=?). Layout must be {COHORTS_ROOT}/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016–2019). Sync cohorts to COHORTS_ROOT if needed, then re-run this cell.")
        return
    print(f"  Location: {path}")
    con = duckdb.connect()
    try:
        dist = con.execute("SELECT target, COUNT(*)::BIGINT AS n FROM read_parquet(?) GROUP BY target ORDER BY target", [str(path)]).fetchall()
        total = sum(row[1] for row in dist)
        by_target = {int(row[0]): int(row[1]) for row in dist}
        n_controls = by_target.get(0, 0)
        n_cases = by_target.get(1, 0)
        ratio = (n_controls / n_cases) if n_cases else 0
        print(f"  Target distribution: {by_target} (total rows: {total:,})")
        print(f"  Control:case ratio: {n_controls:,}:{n_cases:,} = {ratio:.2f}:1")
    finally:
        con.close()

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 4: {cohort} / {age_band} (building model_events.parquet)")
        r = subprocess.run(
            [sys.executable, "create_model_data.py", "--cohort", cohort, "--age-band", age_band],
            cwd=PROJECT_ROOT / "4_model_data",
            capture_output=False,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
        _log_model_data_qa(cohort, age_band)
print("Step 4 complete.")

# Pipeline Phase 5: PGx analysis

Add PGx features (e.g. CPIC drug counts) to model data. Reads from Step 4 outputs and writes updated model data used by Step 6. Run for each cohort/age_band.

In [ ]:
# Pipeline Step 5: run_analysis.py for each REQUIRED_COHORTS (cohort, age_band)
# Set FORCE_STEP5 = True to re-run even when S3 outputs or checkpoints exist
FORCE_STEP5 = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 5: {cohort} / {age_band}")
        cmd = [sys.executable, "run_analysis.py", "--cohort-name", cohort, "--age-band", age_band]
        if FORCE_STEP5:
            cmd.append("--force")
        r = subprocess.run(cmd, cwd=PROJECT_ROOT / "5_pgx_analysis")
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 5 complete.")

# Pipeline Phase 6: Final model deployment

Train final models per cohort/age_band. Reads Step 4 model data and Step 5 PGx features; writes trained models and `feature_schema.json` to `6_final_model/outputs` (or DATA_ROOT). These outputs are used by "Prepare models" and deployment below.

### Pipeline Step 6: Train models

Run **run_final_model.py** for each cohort/age_band. Reads Step 4 model data and Step 5 PGx features; writes trained models and `feature_schema.json` to `6_final_model/outputs`. These outputs are used by "Prepare models" and deployment below.

In [ ]:
# Pipeline Step 6: run_final_model.py for each REQUIRED_COHORTS (cohort, age_band)
# Note: script uses --age_band (underscore)
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 6: {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, "run_final_model.py", "--cohort", cohort, "--age_band", age_band],
            cwd=PROJECT_ROOT / "6_final_model",
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 6 complete.")

/home/pgx3874/jupyter-env/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3678: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Step 1a: Generate model metadata (idempotent, checkpoint)

Extract valid codes (drugs, ICD, CPT) from feature importance for dashboard dropdowns. Uses Step 3b `cohort_feature_importance` when available, else Step 3 aggregated. **Checkpoint:** step is skipped if S3 checkpoint exists. Run this before Step 2 (Combine SHAP/FFA) so metadata is ready for deployment.

In [ ]:
import logging
logger = logging.getLogger(__name__)
if check_step_checkpoint_exists("9_dashboard_metadata", "all", "all", logger):
    print("Step 1 (generate metadata) already completed (checkpoint exists). Skipping.")
else:
    r = subprocess.run([sys.executable, "generate_metadata.py", "--all"], cwd=DATA_PREP_DIR)
    if r.returncode == 0:
        save_step_checkpoint("9_dashboard_metadata", "all", "all", logger=logger)
    if r.returncode != 0:
        raise SystemExit(r.returncode)

### Step 1b: SHAP and FFA artifacts

Runs **run_shap_ffa_workflow.py** (PGx risk calculator pattern) per cohort/age_band: ensures Step 7 (SHAP) exists, runs FFA using XGBoost JSON + SHAP from Step 7, writes to `8_ffa_analysis/outputs`, then runs the combine step so `9_risk_dashboard/outputs` has dashboard_data.json and top_causal_factors. The combine step uses **all patients** by default and **parallel workers** (auto from CPU count) for patient explanations. Uses `8_ffa_analysis/ffa_utils.py` and the XGBoost explainer (SHAP-based rule filtering).

In [ ]:
# Run SHAP + FFA workflow (PGx) per cohort/age_band. Skip if checkpoint exists unless FORCE_STEP1B.
FORCE_STEP1B = False  # Set True to re-run even when checkpoint exists
SHAP_FFA_SCRIPT = PROJECT_ROOT / "9_risk_dashboard" / "data_preparation" / "run_shap_ffa_workflow.py"
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        if not FORCE_STEP1B and check_step_checkpoint_exists("9_shap_artifacts", cohort, age_band, logger):
            print(f"Step 1b (SHAP+FFA) already completed for {cohort}/{age_band} (checkpoint). Skipping.")
            continue
        print(f"→ Step 1b (SHAP+FFA): {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, str(SHAP_FFA_SCRIPT), "--cohort", cohort, "--age-band", age_band],
            cwd=PROJECT_ROOT / "9_risk_dashboard" / "data_preparation",
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
        save_step_checkpoint("9_shap_artifacts", cohort, age_band, logger=logger)
print("SHAP + FFA workflow complete.")

## Step 2: Combine SHAP and FFA results

SHAP (Step 7) and FFA (Step 8) are **mandatory outputs**. Run those steps first, then combine results per cohort/age_band for the Causal Analysis tab. The combine script uses **all patients** by default (`--n-patients 0`) and **parallel workers** for patient explanations (`--workers 0` = auto from CPU count). Run this cell if you need to re-combine without re-running Step 1b (e.g. after changing combine options).

In [ ]:
# Required: combine SHAP (Step 7) and FFA (Step 8) results for Causal Analysis tab.
# Uses all patients (--n-patients 0) and auto parallel workers (--workers 0) by default.
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Combine SHAP/FFA: {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, "combine_shap_ffa_results.py", "--cohort", cohort, "--age-band", age_band, "--output-dir", str(PROJECT_ROOT / "9_risk_dashboard" / "outputs"), "--workers", "0"],
            cwd=DATA_PREP_DIR,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("SHAP/FFA combine complete.")

/home/pgx3874/jupyter-env/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3678: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Results inspection

Load and display combined SHAP/FFA outputs per cohort/age_band (from Step 2 combine). Shows top causal/consensus features and combined importance when available. Run Step 2 first.

In [ ]:
# Inspect combined SHAP/FFA results (consensus_features.json, combined_importance.csv, summary_report.txt)
import json
SHAP_FFA_BASE = PROJECT_ROOT / "9_risk_dashboard" / "outputs"
TOP_N = 15

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        age_band_fname = age_band.replace("-", "_")
        out_dir = SHAP_FFA_BASE / cohort / age_band_fname
        if not out_dir.exists():
            print(f"{cohort} / {age_band}: no combined output dir — run Step 2 first.")
            continue
        print(f"\n{'='*60}\n{cohort} / {age_band}\n{'='*60}")
        # Top combined importance
        combined_path = out_dir / "combined_importance.csv"
        if combined_path.exists():
            import pandas as pd
            df = pd.read_csv(combined_path)
            df = df.head(TOP_N)
            print("Top combined importance (SHAP+FFA):")
            print(df.to_string(index=False))
        else:
            print("combined_importance.csv not found")
        # Consensus features (top-k list)
        consensus_path = out_dir / "consensus_features.json"
        if consensus_path.exists():
            with open(consensus_path) as f:
                data = json.load(f)
            features = data.get("consensus_features", data.get("features", []))[:TOP_N]
            print(f"\nTop {min(TOP_N, len(features))} consensus features: {features}")
        summary_path = out_dir / "summary_report.txt"
        if summary_path.exists():
            print("\nSummary (first 500 chars):")
            print(summary_path.read_text()[:500])

### Feature importance display

Show top-N feature importance for a chosen cohort/age_band from Step 6 (final model) or Step 3b (refined). Useful to verify which features drive the model before deployment.

In [ ]:
# Display top feature importance from Step 6 (or Step 3b) for one or all cohort/age_bands
import pandas as pd
FI_TOP_N = 20
# Option: set to (cohort, age_band) to show one band only, or None to show all
SHOW_ONE_ONLY = None  # e.g. ("opioid_ed", "13-24") or None

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        if SHOW_ONE_ONLY and (cohort, age_band) != SHOW_ONE_ONLY:
            continue
        age_band_fname = age_band.replace("-", "_")
        # Prefer Step 6 XGBoost feature importance; fallback to Step 3b cohort FI
        step6_path = FINAL_MODEL_OUTPUTS / cohort / age_band_fname / f"{cohort}_{age_band_fname}_xgboost_feature_importance.csv"
        step3b_path = PROJECT_ROOT / "3b_feature_importance_eda" / "outputs" / cohort / age_band_fname / f"{cohort}_{age_band_fname}_cohort_feature_importance.csv"
        path = step6_path if step6_path.exists() else step3b_path
        if not path.exists():
            print(f"{cohort} / {age_band}: no feature importance file found.")
            continue
        df = pd.read_csv(path)
        if "importance" not in df.columns and "feature" in df.columns:
            df = df.head(FI_TOP_N)
        else:
            imp_col = "importance" if "importance" in df.columns else df.columns[1]
            df = df.nlargest(FI_TOP_N, imp_col)
        print(f"\n{'='*60}\nTop {FI_TOP_N} features — {cohort} / {age_band}\n{'='*60}")
        print(df.to_string(index=False))

## Step 3: Build and deploy risk calculator

Build the Docker image and push to ECR; then update API Gateway/Lambda. Use the deployment script in `9_risk_dashboard/deployment`.

### Verify infrastructure (Docker, ECR, API Gateway)

Before building and deploying, verify Docker is running, AWS credentials can reach ECR, and API Gateway is set up (if already deployed). Same checks as the PGx risk calculator workflow.

In [ ]:
# Docker, ECR, API Gateway checks for PGx dashboard
import subprocess
import os

print(f"\n{'=' * 80}")
print("Verify infrastructure (Docker, ECR, API Gateway)")
print(f"{'=' * 80}\n")

# 1. Docker
print("1. Docker")
print("-" * 40)
try:
    r = subprocess.run(["docker", "ps"], capture_output=True, text=True, timeout=5)
    if r.returncode == 0:
        print("✓ Docker is running")
    else:
        print("⚠ Docker not accessible:", r.stderr.strip() or r.stdout.strip())
        if "permission denied" in (r.stderr or "").lower():
            print("  Linux: sudo usermod -aG docker $USER && newgrp docker")
except FileNotFoundError:
    print("✗ Docker not found — install Docker first")
except Exception as e:
    print(f"⚠ Error: {e}")
print()

# 2. AWS / ECR
print("2. ECR (AWS credentials + repository)")
print("-" * 40)
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
ECR_REPOSITORY = os.environ.get("ECR_REPOSITORY", "pgx-risk-calculator")
try:
    # Get caller identity (proves credentials work)
    r = subprocess.run(
        ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
        capture_output=True, text=True, timeout=10
    )
    if r.returncode != 0:
        print("⚠ AWS CLI not configured or no credentials:", (r.stderr or r.stdout or "").strip())
    else:
        account = r.stdout.strip()
        print(f"✓ AWS identity: account {account}")
    # ECR repository exists?
    r2 = subprocess.run(
        ["aws", "ecr", "describe-repositories", "--repository-names", ECR_REPOSITORY, "--region", AWS_REGION],
        capture_output=True, text=True, timeout=10
    )
    if r2.returncode == 0:
        print(f"✓ ECR repository exists: {ECR_REPOSITORY}")
    else:
        print(f"⚠ ECR repository '{ECR_REPOSITORY}' not found (create it or set ECR_REPOSITORY)")
        print("  docker_build.sh can create it automatically on first push")
except FileNotFoundError:
    print("✗ AWS CLI not found")
except Exception as e:
    print(f"⚠ Error: {e}")
print()

# 3. API Gateway
print("3. API Gateway")
print("-" * 40)
print("Note: API Gateway should already be set up; this step only verifies.")
API_NAME = os.environ.get("PGX_API_GATEWAY_NAME", "pgx-risk-calculator-api")
try:
    r = subprocess.run(
        ["aws", "apigateway", "get-rest-apis", "--query", f"items[?name=='{API_NAME}'].id", "--output", "text"],
        capture_output=True, text=True, timeout=10
    )
    if r.returncode == 0 and r.stdout.strip():
        api_id = r.stdout.strip().split()[0]
        api_url = f"https://{api_id}.execute-api.{AWS_REGION}.amazonaws.com/prod"
        print(f"✓ API Gateway found: {API_NAME}")
        print(f"  API ID: {api_id}")
        print(f"  Base URL: {api_url}")
        # Optional: test endpoint (e.g. metadata)
        try:
            r2 = subprocess.run(
                ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", f"{api_url}/metadata"],
                capture_output=True, text=True, timeout=5
            )
            if r2.returncode == 0 and r2.stdout.strip() in ("200", "301", "302"):
                print("  ✓ API is responding")
            else:
                print("  ⚠ API endpoint test inconclusive (curl or endpoint may differ)")
        except Exception:
            print("  (skipping endpoint test — curl not available)")
    else:
        print(f"⚠ API Gateway '{API_NAME}' not found or AWS CLI not configured")
        print("  Set up API Gateway and link to Lambda, then re-run this check.")
except Exception as e:
    print(f"⚠ Could not verify API Gateway: {e}")
    print("  Assuming API Gateway is already set up or will be configured after deploy.")

print(f"\n{'=' * 80}")

### Prepare Lambda directory

Assemble `lambda_dir` under `9_risk_dashboard` for Docker build (models, metadata, CPIC data).

In [ ]:
subprocess.run([sys.executable, "prepare_lambda_dir.py", "--verify-only"], cwd=DEPLOY_DIR, check=True)

In [ ]:
# Run from DEPLOY_DIR so prepare_lambda_dir.py finds paths correctly (no shell var expansion in notebook)
r = subprocess.run([sys.executable, "prepare_lambda_dir.py"], cwd=DEPLOY_DIR)
if r.returncode != 0:
    raise SystemExit(r.returncode)
print("Lambda directory prepared.")

### Prepare models — idempotent with checkpoint

Package models and feature schemas from `6_final_model/outputs` into `9_risk_dashboard/outputs/models`. **Checkpoint:** step is skipped if S3 checkpoint exists.

In [ ]:
import logging
logger = logging.getLogger(__name__)
if check_step_checkpoint_exists("9_dashboard_models", "all", "all", logger):
    print("Step 3 (prepare models) already completed (checkpoint exists). Skipping.")
else:
    r = subprocess.run([sys.executable, "prepare_models.py", "--all"], cwd=DATA_PREP_DIR)
    if r.returncode == 0:
        save_step_checkpoint("9_dashboard_models", "all", "all", logger=logger)
    if r.returncode != 0:
        raise SystemExit(r.returncode)

### Docker Build

In [ ]:
# Variables for Docker build (DASHBOARD_DIR from setup cell)
risk_dashboard_dir = DASHBOARD_DIR
needs_prepare = True  # Set True to prompt for build; False to skip rebuild

print(f"\n{'=' * 80}")
print("Step 3: Build and Push Docker Image")
print(f"{'=' * 80}")

docker_script = risk_dashboard_dir / "docker_build_pgx.sh"

needs_docker_build = needs_prepare

if docker_script.exists():
    print(f"\nDocker build strategy:")
    print(f"  - Lambda directory was {'updated' if needs_prepare else 'unchanged'}")
    print(f"  - Docker image will be {'built' if needs_docker_build else 'skipped (use --force to rebuild)'}")
    print("-" * 80)
    print("\nThis will:")
    print("  1. Build Docker image with models and dependencies")
    print("  2. Push image to AWS ECR (Elastic Container Registry)")
    print("-" * 80)
    print("\n⚠ Note: This requires:")
    print("  - Docker installed and running")
    print("  - AWS CLI configured with ECR permissions")
    print("  - AWS credentials with push access to ECR")
    print("-" * 80)
    
    if needs_docker_build:
        response = input("\nProceed with Docker build? (y/n): ").strip().lower()
    else:
        print("\n⏭ Skipping Docker build (Lambda directory unchanged)")
        print("  To force rebuild, run: ./docker_build_pgx.sh --force")
        response = 'n'
    
    if response == 'y':
        try:
            result = subprocess.run(
                ["bash", str(docker_script)],
                cwd=str(risk_dashboard_dir),
                capture_output=False,
                text=True
            )
            
            if result.returncode == 0:
                print(f"\n✓ Docker image built and pushed successfully!")
                print(f"\n  Next: Get ECR URI from output above and use it to update Lambda")
            else:
                print(f"\n⚠ Docker build exited with code: {result.returncode}")
        except Exception as e:
            print(f"\n✗ Error building Docker image: {e}")
            logger.error("Error building Docker image", exc_info=True)
else:
    print(f"\n⚠ Docker build script not found: {docker_script}")
    print("  Expected location: 9_risk_dashboard/docker_build_pgx.sh")

print(f"\n{'=' * 80}")

### Update Lambda

In [ ]:
# Step 4: Update Lambda Function (Idempotent - only if Docker image was updated)
print(f"\n{'=' * 80}")
print("Step 4: Update Lambda Function")
print(f"{'=' * 80}")

lambda_function_name = "pgx-risk-calculator"
region = "us-east-1"

# Check if Lambda function exists
lambda_exists = False
try:
    result = subprocess.run(
        ["aws", "lambda", "get-function", "--function-name", lambda_function_name, "--region", region],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        lambda_exists = True
        print(f"\n✓ Lambda function exists: {lambda_function_name}")
except:
    print(f"\n⚠ Could not check Lambda function status")
    print("  (AWS CLI may not be configured)")

if lambda_exists and needs_docker_build:
    print(f"\nLambda function will be updated with new Docker image")
    print("-" * 80)
    print("\n⚠ Note: AWS credentials are automatically used from EC2 instance role")
    print("  (No need to run 'aws configure' if instance has IAM role attached)")
    print("-" * 80)
    
    # Get AWS account ID and ECR URI
    try:
        result = subprocess.run(
            ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode == 0:
            account_id = result.stdout.strip()
            ecr_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/pgx-risk-calculator:latest"
            print(f"\n  AWS Account ID: {account_id}")
            print(f"  ECR URI: {ecr_uri}")
            
            # Also get identity to show which role is being used
            identity_result = subprocess.run(
                ["aws", "sts", "get-caller-identity", "--output", "json"],
                capture_output=True,
                text=True,
                timeout=5
            )
            if identity_result.returncode == 0:
                import json
                identity = json.loads(identity_result.stdout)
                if "Arn" in identity:
                    print(f"  Using IAM Role: {identity['Arn']}")
            
            response = input("\nProceed with Lambda update? (y/n): ").strip().lower()
            
            if response == 'y':
                try:
                    result = subprocess.run(
                        [
                            "aws", "lambda", "update-function-code",
                            "--function-name", lambda_function_name,
                            "--image-uri", ecr_uri,
                            "--region", region
                        ],
                        capture_output=False,
                        text=True
                    )
                    
                    if result.returncode == 0:
                        print(f"\n✓ Lambda function updated successfully!")
                        print(f"\n  Waiting for update to complete...")
                        # Wait for function to be ready
                        subprocess.run(
                            ["aws", "lambda", "wait", "function-updated",
                             "--function-name", lambda_function_name,
                             "--region", region],
                            capture_output=True
                        )
                        print(f"  ✓ Lambda function is ready")
                    else:
                        print(f"\n⚠ Lambda update exited with code: {result.returncode}")
                except Exception as e:
                    print(f"\n✗ Error updating Lambda: {e}")
                    logger.error("Error updating Lambda", exc_info=True)
            else:
                print("\n⏭ Skipping Lambda update")
        else:
            print("\n⚠ Could not retrieve AWS Account ID")
    except:
        print("\n⚠ Could not retrieve AWS Account ID - check AWS CLI configuration")
elif not lambda_exists:
    print(f"\n⚠ Lambda function '{lambda_function_name}' does not exist")
    print("  Create it first using AWS Console or CLI")
elif not needs_docker_build:
    print(f"\n⏭ Skipping Lambda update (Docker image unchanged)")
else:
    print("\nTo update Lambda function manually:")
    print("-" * 80)
    print("\n1. Get your AWS Account ID:")
    print("   AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)")
    print("\n2. Construct ECR URI:")
    print("   ECR_URI=\"${AWS_ACCOUNT_ID}.dkr.ecr.us-east-1.amazonaws.com/pgx-risk-calculator:latest\"")
    print("\n3. Update Lambda function:")
    print("   aws lambda update-function-code \\")
    print("       --function-name pgx-risk-calculator \\")
    print("       --image-uri ${ECR_URI} \\")
    print("       --region us-east-1")

print(f"\n{'=' * 80}")

### Update HTML/js Front End on S3

In [ ]:
# Step 6: Upload HTML to S3 (Idempotent - only if HTML changed)
print(f"\n{'=' * 80}")
print("Step 5: Upload HTML Dashboard to S3")
print(f"{'=' * 80}")

html_file = risk_dashboard_dir / "pgx_dashboard.html"
s3_bucket = "jerome-dixon.io"  # Update for VCU or your bucket
s3_prefix = "vcu/pgx-risk-calculator"
s3_path = f"s3://{s3_bucket}/{s3_prefix}/index.html"

if html_file.exists():
    print(f"\nHTML file found: {html_file}")
    
    # Check if S3 file exists and compare modification times
    needs_upload = True
    try:
        result = subprocess.run(
            ["aws", "s3", "ls", s3_path, "--region", "us-east-1"],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        if result.returncode == 0 and result.stdout.strip():
            # S3 file exists - check if local is newer
            local_mtime = html_file.stat().st_mtime
            
            # Parse S3 last modified time from ls output
            # Format: "2026-01-26 10:30:45    12345 index.html"
            s3_output = result.stdout.strip()
            if s3_output:
                print(f"\n✓ S3 file exists")
                print(f"  Checking if local file is newer...")
                
                # Get S3 file metadata
                head_result = subprocess.run(
                    ["aws", "s3api", "head-object", "--bucket", s3_bucket, 
                     "--key", f"{s3_prefix}/index.html", "--region", "us-east-1"],
                    capture_output=True,
                    text=True,
                    timeout=5
                )
                
                if head_result.returncode == 0:
                    import json
                    s3_meta = json.loads(head_result.stdout)
                    s3_mtime_str = s3_meta.get("LastModified", "")
                    if s3_mtime_str:
                        from datetime import datetime
                        s3_mtime = datetime.fromisoformat(s3_mtime_str.replace("Z", "+00:00")).timestamp()
                        
                        if local_mtime <= s3_mtime:
                            print(f"  ✓ S3 file is up to date (local: {local_mtime}, S3: {s3_mtime})")
                            needs_upload = False
                        else:
                            print(f"  ⚠ Local file is newer - will upload")
        else:
            print(f"\n⚠ S3 file not found - will upload")
    except:
        print(f"\n⚠ Could not check S3 file status (AWS CLI may not be configured)")
        print(f"  Will attempt upload")
    
    if needs_upload:
        print(f"\nUploading to S3:")
        print("-" * 80)
        print(f"  Source: {html_file}")
        print(f"  Destination: {s3_path}")
        print("-" * 80)
        print("\n⚠ Note: This requires:")
        print("  - AWS CLI configured")
        print("  - S3 write permissions")
        print("  - Bucket exists and is accessible")
        print("-" * 80)
        
        response = input("\nProceed with S3 upload? (y/n): ").strip().lower()
        
        if response == 'y':
            try:
                result = subprocess.run(
                    [
                        "aws", "s3", "cp",
                        str(html_file),
                        s3_path,
                        "--content-type", "text/html",
                        "--cache-control", "no-cache",
                        "--region", "us-east-1"
                    ],
                    capture_output=False,
                    text=True
                )
                
                if result.returncode == 0:
                    print(f"\n✓ HTML uploaded successfully to S3!")
                    print(f"\n  Dashboard URL: https://{s3_bucket}/{s3_prefix}/")
                else:
                    print(f"\n⚠ S3 upload exited with code: {result.returncode}")
            except Exception as e:
                print(f"\n✗ Error uploading to S3: {e}")
                logger.error("Error uploading to S3", exc_info=True)
        else:
            print("\n⏭ Skipping S3 upload")
    else:
        print(f"\n⏭ Skipping S3 upload (file is up to date)")
else:
    print(f"\n⚠ HTML file not found: {html_file}")
    print("  Expected location: 9_risk_dashboard/pgx_dashboard.html")

print(f"\n{'=' * 80}")

# Shutdown EC2

In [ ]:
# Set SHUTDOWN_EC2 = True to enable, False to disable
SHUTDOWN_EC2 = True  # Change to True to enable auto-shutdown

print(f"\n{'=' * 80}")
print("Final Step: EC2 Instance Shutdown (Optional)")
print(f"{'=' * 80}")

if SHUTDOWN_EC2:
    print("\nShutting down EC2 instance...")
    print("-" * 80)
    
    import subprocess
    import shutil
    import os
    
    # Get instance ID from EC2 metadata service
    try:
        result = subprocess.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True,
            text=True,
            timeout=5
        )
        instance_id = result.stdout.strip()
        
        if instance_id and len(instance_id) > 0:
            print(f"Instance ID: {instance_id}")
            
            # Find AWS CLI
            aws_cmd = shutil.which("aws")
            if not aws_cmd:
                # Try common paths
                aws_paths = [
                    "/usr/local/bin/aws",
                    "/usr/bin/aws",
                    "/home/ec2-user/.local/bin/aws"
                ]
                for path in aws_paths:
                    if os.path.exists(path):
                        aws_cmd = path
                        break
            
            if aws_cmd:
                # Stop the instance (use terminate-instances for permanent deletion)
                shutdown_cmd = [aws_cmd, "ec2", "stop-instances", "--instance-ids", instance_id]
                
                print(f"Running: {' '.join(shutdown_cmd)}")
                result = subprocess.run(shutdown_cmd, capture_output=True, text=True)
                
                if result.returncode == 0:
                    print("\n✓ EC2 instance stop command sent successfully")
                    print("Instance will stop in a few moments.")
                    print("Note: This is a STOP (not terminate), so you can restart it later.")
                    logger.info(f"EC2 instance {instance_id} stop command sent successfully")
                else:
                    print(f"\n⚠ EC2 stop command returned exit code {result.returncode}.")
                    print("Check AWS credentials and permissions.")
                    if result.stderr:
                        print(f"Error: {result.stderr}")
                    logger.warning(f"EC2 stop command failed: {result.stderr}")
            else:
                print("\nWarning: AWS CLI not found. Cannot shutdown instance.")
                print("Install AWS CLI or ensure it's in your PATH.")
                logger.warning("AWS CLI not found, cannot shutdown EC2 instance")
        else:
            print("\nWarning: Could not determine instance ID. Skipping shutdown.")
            print("If you want to shutdown manually, use:")
            print("  aws ec2 stop-instances --instance-ids <your-instance-id>")
            logger.warning("Could not determine EC2 instance ID")
    except subprocess.TimeoutExpired:
        print("\nWarning: Timeout retrieving instance ID from metadata service.")
        print("If running on EC2, check that metadata service is accessible.")
        logger.warning("Timeout retrieving EC2 instance ID from metadata service")
    except Exception as e:
        print(f"\nWarning: Could not retrieve instance ID: {e}")
        print("If you want to shutdown manually, use:")
        print("  aws ec2 stop-instances --instance-ids <your-instance-id>")
        logger.warning(f"Error retrieving EC2 instance ID: {e}")
else:
    print("\nEC2 Auto-Shutdown: DISABLED")
    print("To enable auto-shutdown, set SHUTDOWN_EC2 = True in this cell.")
    print("Instance will continue running.")

print(f"\n{'=' * 80}")
print("Workflow Complete!")
print(f"{'=' * 80}")